# **Agentic RAG Pipeline**

### Content Outline 
1. Introduction 
2. Agentic RAG Pipeline: Getting Started 
3. Step 1: Loading the Knowledge
4. Step 2: Chunking
5. Step 3: Embeddings & Vector Store
6. Step 4: The Brain 
7. Step 5: The Agent 
8. Step 6: The Execution Loop 
9. Closing Thoughts


### Introduction

A typical RAG system quickly searches your uploaded tax documents or technical manuals to find a relevant snippet, even for something as simple as a casual greeting. **Agentic RAG** works differently. Instead of always looking up information, the agent stops to consider whether it really needs to search or if it can answer on its own. 

In this article, I'll show you how to build an **Agentic RAG pipeline** using **Python** and **LangChain**.  

### Agentic RAG Pipeline: Getting Started 

In this guide, we'll build a local, privacy-friendly Agentic RAG pipeline using Python, LangChain, and a lightweight Google model. We'll go beyond just writing code to create a system that acts a bit more like a person. 

We'll use **LangChain** to manage the process, **ChromaDB** for storing vectors, and **Google's Flan-T5** as our local language model. Everything runs on your own computer, so you don't need any API keys. 

You will need to **install a few libraries**. In your **terminal**: 

In [ ]:
pip install langchain langchain-community langchain-chroma transformers sentence-transformers pypdf

### Step 1: Loading the Knowledge

First, we need to give our AI something to read. We use a function to scan a folder for PDFs:

In [1]:
import os 
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline 

# Load PDFs from a folder 
def load_docs(folder_path):
    docs = []
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(folder_path, file))
            docs.extend(loader.load())
    return docs

# Update this path to where your PDFs are stored 
docs = load_docs("/Users/yanbingjiang/Desktop/ai-engineering-projects-2026/03-agentic-rag-pipeline/data")
print("PDF Pages Loaded:", len(docs))

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PDF Pages Loaded: 290


In this step, we go through a folder, find PDF files, and load them one page at a time. It's like stacking on your desk before you start studying.

### Step 2: Chunking 

LLMs can only read a certain amount of text at once, called the context window. Even if they could handle more, giving them a whole 500-page book to answer one question isn't efficient: 

In [2]:
# Split PDFs into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=80
)
chunks = text_splitter.split_documents(docs)
print("Chunks Created:", len(chunks))

Chunks Created: 2115


Notice **chunk_overlap=80**. Instead of just cutting the text, we let the chunks **overlap** a bit. This way, sentences aren't split in half at the edge of a chunk, so the meaning (or semantic context) is preserved across breaks. 

### Step 3: Embeddings & Vector Store

Now we convert text into numbers, called **vectors**, that the computer can understand. We store these in **Chroma**, which is a **vector database**:

In [3]:
# Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Save texts into Chroma vector DB
texts = [c.page_content for c in chunks]

# Guard: don't build DB if no documents loaded
if not texts:
    print("⚠️  No documents found. Add PDFs to your data folder first.")
    print("Folder path:", "/Users/yanbingjiang/Desktop/ai-engineering-projects-2026/03-agentic-rag-pipeline/data")
else:  
    db = Chroma.from_texts(
        texts=texts,
        embedding=embedding_model,      
        collection_name="rag_store"
    )

# Retriever
    retriever = db.as_retriever(search_kwargs={"k": 3})
    print("Vector DB ready. Total docs:", db._collection.count())

/var/folders/sq/4nfvmtr14g33p36_f9k7gdf00000gn/T/ipykernel_18963/2528705947.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Vector DB ready. Total docs: 2115


In [4]:
# Run this cell alone to verify embeddings work
test = embedding_model.embed_query("hello world")
print("Embedding length:", len(test))  # Should print 384, not 0

Embedding length: 384


We're using **all-MiniLM-L6-v2**. It's a small, fast model that's great for local development. It **puts similar concepts close together** in space. When we search later, we won't be matching keywords; we'll be **matching meanings**. 

### Step 4: The Brain 

We need a model to **generate the actual answers**. We are using **google/flan-t5-base**:

In [5]:
# Local LLM 
llm = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=150
)

Device set to use mps:0


**Flan-T5** is a "seq2seq" model. It's **great at** following instructions like "Summarize this" or "Answer this," which makes it ideal for RAG tasks even though it's smaller.

### Step 5: The Agent

This is the key part. **This simple function is what makes the pipeline Agentic**:

In [6]:
# Agent brain 
def agent_controller(query):
    q = query.lower()
    if any(word in q for word in ["pdf", "document", "data", "summarize", "information", "find"]):
        return "search"
    return "direct"

Instead of sending everything to the database, this controller analyzes the user's intent: 
1. Does this user want data from the file? **Action: Search**
2. Is the user just chatting or asking for general knowledge? **Action: Direct**

In a production system, you might use a powerful LLM to make this decision. But for learning, this keyword-based method is a great way to show the **Routing Pattern** in agentic AI. 

### Step 6: The Execution Loop 

Finally, we tie it all together:

In [7]:
# Define retriever safely — None if DB cell was never run
retriever = globals().get("retriever", None)

In [12]:
# RAG
def rag_answer(query):
    action = agent_controller(query)

    if action == "search":
        if retriever is None:
            print("⚠️  Retriever not available — no documents loaded. Falling back to direct answer.")
            final_prompt = query
        else:
            print(f"🕵️ Agent decided to SEARCH document for: '{query}'")
            results = retriever.invoke(query)
            context = "\n".join([r.page_content for r in results])
            final_prompt = f"Use this context:\n{context}\n\nAnswer:\n{query}"
    else:
        print(f"🤖 Agent decided to answer DIRECTLY: '{query}'")
        final_prompt = query

    response = llm(final_prompt)[0]["generated_text"] 
    return response


In [14]:
# Test 1: A document-specific question
query = "Give me a 5-point summary from the PDF"
print(rag_answer(query))

print("-" * 20)

# Test 2: A general knowledge question 
print(rag_answer("What is an Ideal Resume Format? Explain in 50 words."))

🕵️ Agent decided to SEARCH document for: 'Give me a 5-point summary from the PDF'
Broadening digital access 53% 60% Increased focus on labour and social issues 47% 46% Slower economic growth 47% 42% Increased efforts and investments to adapt to climate... 44% 41% Increased geopolitical division and conflicts 44% 34% Increased government subsidies and industrial policy 24% 21% Increased focus on labour and social issues 35% 46% Growing working-age populations 25% 24% Slower economic growth 25% 42% Ageing and declining working-age populations 20% 40% Increased restrictions to global trade and investment 15% 23% Stricter anti-trust and competition regulations 5% 17% Prices or inflation 45% 50% Slower economic growth 45% 42% Ageing and declining working- age populations 
--------------------
🤖 Agent decided to answer DIRECTLY: 'What is an Ideal Resume Format? Explain in 50 words.'
An Ideal Resume Format is a format for a resume to be written in a professional manner. An Ideal Resume Format

In the **first** case, the agent sees the word "PDF" or "summary" and use the retriever. In the **second** case, it knows it doesn't need your documents to explain a resume format, so it answers using its own pre-trained knowledge. 

### Closing Thoughts

When I first started working with AI, I thought bigger was better. I wanted the largest model and the biggest database. But over time, I learned that **intelligence is really about efficiency, not just raw power.**

By building this Agentic router, you save computing resources and reduce latency. Most importantly, you create a system that respects the user's context. 
